# Imports

In [2]:
import numpy as np
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn

# Neural Network

In [3]:
# define custom NN class
# lets PyTorch track parameters and gradients automatically
class PINN(nn.Module):
    def __init__(self): # initializes nn.Module
        super().__init__()
        self.net = nn.Sequential( # sequential container for layers, = feed-forward pipeline
            nn.Linear(1, 20), # input = 1 number (x), output = 20 hidden features
            nn.Tanh(),
            # add more layers for more depth, better fxn approximation
            nn.Linear(20, 20),
            nn.Tanh(),
            nn.Linear(20, 1) # output = 1 number (u)
        )

    # forward method defines how input data flows through the network
    def forward(self, x):
        return self.net(x)

## Physics Loss

In [4]:
# define a fxn to enforce the physics constraint (ODE)
# how wrong is the NN at satisfying the ODE? want to minimize this loss
def physics_loss(model, x):
    x.requires_grad = True # track gradients w.r.t. x for autograd
    u = model(x) # NN's prediction for u(x)

    du_dx = torch.autograd.grad(
        u, x,
        grad_outputs=torch.ones_like(u), # seed vector for gradient (du/dx)
        create_graph=True # allows higher-order derivatives if needed
    )[0]

    # ODE: du/dx + u = 0
    return torch.mean((du_dx + u)**2)

## Boundary Condition Loss

In [5]:
# define a fxn to enforce the boundary condition(s)
def boundary_loss(model):
    x0 = torch.tensor([[0.0]]) # evaluate at x=0
    u0 = model(x0) # network's prediction at boundary pt, u(0)
    return (u0 - 1.0)**2 # want u(0)=1, so penalize deviation from 1.0

# Training

In [6]:
model = PINN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# train for many iterations (epochs)
for epoch in range(5000):
    optimizer.zero_grad() # reset gradients before backprop

    x = torch.rand((100, 1))  # collocation points: random x in [0,1] where physics is enforced

    loss_p = physics_loss(model, x) # enforces diff eqn everywhere
    loss_b = boundary_loss(model) # enforces BC at specific point(s)

    loss = loss_p + loss_b # loss=physics+BC
    loss.backward() # backpropagation: computes gradients of all parameters
    optimizer.step() # updates network weights

    if epoch % 500 == 0: # track convergence every 500 epochs
        print(f"Epoch {epoch}, Loss: {loss.item()}")

/home/jkreinsen/projects/pinn/venv/lib/python3.10/site-packages/torch/autograd/graph.py:869: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 0, Loss: 1.7984117269515991
Epoch 500, Loss: 0.0001694163220236078
Epoch 1000, Loss: 6.776945519959554e-05
Epoch 1500, Loss: 3.2466337870573625e-05
Epoch 2000, Loss: 2.0106655938434415e-05
Epoch 2500, Loss: 5.847789452673169e-06
Epoch 3000, Loss: 3.83467295250739e-06
Epoch 3500, Loss: 1.2217303719808115e-06
Epoch 4000, Loss: 7.981975613802206e-07
Epoch 4500, Loss: 9.288552860198251e-07


# Evaluation

## RMSE

In [7]:
# using numpy
y_true = np.array([10, 12, 14, 16])
y_pred = np.array([11, 11, 15, 15])

rmse = np.sqrt(np.mean((y_pred - y_true)**2))
print(rmse)

1.0


In [10]:
# using mean_squared_error from sklearn.metrics

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(rmse)

1.0


In [9]:
# # using torch
# y_pred = model(x)

# rmse = torch.sqrt(torch.mean((y_pred - y_true)**2)) # = torch.sqrt(loss)
# print(rmse.item())